# 05 - Clustering

**Based on:** *AI in Chemical Engineering: Unlocking the Power Within Data* (Romagnoli et al.)

My learning notes and implementations on clustering methods:
- K-Means
- Elbow Method and Silhouette Score
- DBSCAN
- HDBSCAN
- Hierarchical Clustering
- A simple preprocessing + clustering pipeline


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

np.random.seed(42)
n = 300
df = pd.DataFrame({
    'temperature': np.random.normal(85, 6, n),
    'pressure': np.random.normal(2.5, 0.3, n),
    'flow_rate': np.random.normal(120, 12, n),
    'concentration': np.random.normal(0.45, 0.06, n)
})

scaler = StandardScaler()
df_scaled = scaler.fit_transform(df)

print("Hazır, şekil:", df_scaled.shape)

Centroid-based → like KMeans
- You specify the number of clusters.
- It assumes the clusters are round / spherical.
- Each cluster has a "center".

Density-based → DBSCAN and HDBSCAN
- You do not specify the number of clusters.
- They do not make an assumption about cluster shape.
- Dense regions become clusters, sparse regions can become noise.


Elbow → "How many clusters make sense?"

Silhouette → "How good are those clusters?"


KMeans is the most classic clustering method.

What is it trying to do?
It tries to divide the data into the K groups that you specify.

How does it work? (very roughly):

1. You say, "I want 3 clusters" (K=3).
2. The algorithm randomly chooses 3 center points (centroids).
3. It assigns each data point to the nearest center.
4. Then it updates the centers.
5. It repeats this process until the centers stabilize.


What is its biggest problem?

You have to specify K yourself.
The algorithm does not know "how many clusters should there be?"

So today we will:
- Try different K values.
- Use the Elbow method to see "which K makes the most sense?"
- Use the Silhouette score to measure how well the clusters are separated.


In [ ]:
# Create the KMeans model
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)

# Fit + predict
clusters = kmeans.fit_predict(df_scaled)

# Let's see the results
print("Cluster numbers of the first 20 points:")
print(clusters[:20])

print("\nNumber of points in each cluster:")
print(pd.Series(clusters).value_counts().sort_index())

What does `fit_predict` mean?

There are two separate jobs in KMeans:

**fit (training)**  
The algorithm looks at the data and learns:
"Where should the centers of these 3 clusters be?"
So it finds the centroids.

**predict**  
It asks each point, "Which cluster do you belong to?"
and assigns a number such as 0, 1, or 2.

`fit_predict` does both at once.


`n_clusters=3` → I want 3 clusters.

`random_state=42` → keep the result reproducible.

`n_init=10` → try 10 different initializations and choose the best one.

`fit_predict` → train the model and assign a cluster number to every point.


In [ ]:
# Reduce to 2 dimensions with PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(df_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(pca_result[:, 0], pca_result[:, 1], c=clusters, cmap='viridis', alpha=0.7)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("KMeans (K=3) - PCA visualization")
plt.colorbar(label="Cluster")
plt.grid(True)
plt.show()

## Elbow Method and Silhouette Score

Elbow asks how many clusters make sense; Silhouette asks how good those clusters are.


In [ ]:
inertias = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(df_scaled)
    inertias.append(km.inertia_)
plt.plot(K_range, inertias, 'bo-')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.grid(True)
plt.show()

In [ ]:
silhouette_scores = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(df_scaled)
    score = silhouette_score(df_scaled, labels)
    silhouette_scores.append(score)
    print(f"K={k} → Silhouette Score: {score:.3f}")
plt.plot(K_range, silhouette_scores, 'go-')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Choosing K with Silhouette Score")
plt.grid(True)
plt.show()

# DBSCAN

KMeans requires K, assumes roughly spherical clusters, and does not explicitly identify noise. DBSCAN can discover irregularly shaped clusters and mark outliers as noise.

### Core point
A point with at least `MinPts` neighbors inside the `eps` radius.

### Border point
Not a core point itself, but close to a core point.

### Noise point
Neither core nor border. DBSCAN labels these points as `-1`.


In [ ]:
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN

X, y = make_moons(n_samples=300, noise=0.08, random_state=42)
Xscaled = StandardScaler().fit_transform(X)
db = DBSCAN(eps=0.3, min_samples=5)
clusters = db.fit_predict(Xscaled)
n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
n_noise = list(clusters).count(-1)
print(f"Number of clusters found: {n_clusters}")
print(f"Noise points: {n_noise}")
plt.scatter(Xscaled[:, 0], Xscaled[:, 1], c=clusters)
plt.title("DBSCAN")
plt.show()

# HDBSCAN

HDBSCAN is a hierarchical version of DBSCAN. It can handle clusters with different densities more effectively and usually needs less manual density-threshold tuning.

### Core Distance
How far must we go around a point to find the required number of neighbors? Dense regions have smaller core distances.

### Mutual Reachability Distance
HDBSCAN modifies ordinary distance using core distances, making density differences easier to handle.

### Hierarchy and stability
It builds clusters across different density levels and tries to retain stable clusters.


In [ ]:
# Install once if needed: pip install hdbscan
import hdbscan
hdb = hdbscan.HDBSCAN(min_cluster_size=15, min_samples=5)
clusters_hdb = hdb.fit_predict(Xscaled)
n_clusters_hdb = len(set(clusters_hdb)) - (1 if -1 in clusters_hdb else 0)
n_noise_hdb = list(clusters_hdb).count(-1)
print(f"HDBSCAN - Number of clusters: {n_clusters_hdb}")
print(f"HDBSCAN - Noise points: {n_noise_hdb}")
plt.scatter(Xscaled[:, 0], Xscaled[:, 1], c=clusters_hdb)
plt.title("HDBSCAN Result")
plt.show()

# Hierarchical Clustering

Difference from KMeans:
- You can choose the number of clusters by examining the hierarchy / dendrogram.
- Different linkage criteria are available (`ward`, `complete`, `average`, `single`).

`ward` merges clusters while trying to minimize the increase in within-cluster variance.


In [ ]:
from sklearn.cluster import AgglomerativeClustering
agg = AgglomerativeClustering(n_clusters=2, linkage='ward')
clusters_agg = agg.fit_predict(Xscaled)
print("Number of hierarchical clusters:", len(set(clusters_agg)))
plt.scatter(Xscaled[:, 0], Xscaled[:, 1], c=clusters_agg)
plt.title("Hierarchical Clustering (n_clusters=2)")
plt.show()

| Method | Specify number of clusters? | Detects noise? | Shape assumption | Useful for |
|---|---|---|---|---|
| KMeans | Yes | No | Roughly spherical | Simple, compact clusters |
| DBSCAN | No | Yes | No fixed shape | Irregular clusters + outliers |
| HDBSCAN | No | Yes | No fixed shape | Different-density clusters |
| Hierarchical | Selected from hierarchy | No | Depends on linkage | Nested cluster structure |


# Pipeline

A `Pipeline` combines preprocessing and modeling steps into one object. Here: `StandardScaler → PCA → KMeans`.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_blobs

X, _ = make_blobs(n_samples=300, centers=3, random_state=42)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=2)),
    ('kmeans', KMeans(n_clusters=3, random_state=42, n_init=10))
])
clusters = pipe.fit_predict(X)
print(pd.Series(clusters).value_counts().sort_index())
X_pca = pipe.named_steps['pca'].transform(pipe.named_steps['scaler'].transform(X))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='viridis', alpha=0.8)
plt.title("Pipeline: Scale + PCA + KMeans")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(label="Cluster")
plt.grid(True)
plt.show()